# Glowtrace dataset collector

Finds and downloads every skin, hair and nail dataset we're allowed to use commercially.

It runs in three steps:
1. The curated Roboflow sets.
2. A fresh search of Hugging Face, Zenodo, Figshare, Wikimedia Commons, Roboflow and Kaggle, using terms focused on our users: braids, edges, natural hair, melasma, deeper skin tones and manicures.
3. Every find that is licensed for commercial use, relevant and a sensible size.

Everything lands in your Google Drive (**Glowtrace/data**), so the training notebook can reuse it without downloading again.

**What to do:**
1. No GPU needed.
2. Check `ROBOFLOW_API_KEY` is in Secrets. Kaggle (`KAGGLE_USERNAME`, `KAGGLE_KEY`) is optional.
3. **Runtime → Run all.** It takes 30-90 minutes, depending on what it finds.
4. Send Claude `credits.json` and `scout_report.csv` from the last cell.

Hugging Face sets are skipped by default because most don't say where their photos came from. Claude checks every new set's photo sources before any training.

In [ ]:
# 1. Drive, tools and keys. Data is kept in Drive, so a disconnect loses nothing.
import os
from google.colab import drive, userdata
drive.mount('/content/drive')
HOME = '/content/drive/MyDrive/Glowtrace'
os.makedirs(HOME, exist_ok=True)
os.chdir(HOME)
!pip install -q roboflow huggingface_hub kaggle datasets
for name in ['ROBOFLOW_API_KEY', 'KAGGLE_USERNAME', 'KAGGLE_KEY']:
    try:
        os.environ[name] = userdata.get(name); print(name, 'found')
    except Exception:
        print(name, 'not set (that source is skipped)')
print('Working in', os.getcwd())

In [ ]:
%%writefile fetch_datasets.py
"""Download every Glowtrace training dataset from Roboflow Universe into data/<area>/.

Needs a free Roboflow API key in the ROBOFLOW_API_KEY environment variable (in Colab: the key
icon, "Secrets"). For each dataset it reads the project's details, skips anything whose licence
says non-commercial, downloads the newest version's images, and records name, link and licence
in data/credits.json (CC BY requires credit). Failures are reported and skipped, never fatal.

Usage: python3 fetch_datasets.py [skin|nails|hair ...]
"""
import json
import os
import shutil
import sys
import urllib.request

DATASETS = {
    "skin": [
        ("workspace-1-myjxi", "acne-detection-cbm6h", "Acne, dark circles, pores, wrinkles"),
        ("facial-zfrfm", "facial-ngz7q", "13 facial concerns"),
        ("work-acne", "pigmentation-jiao4", "Pigmentation and spots"),
        ("testworkspace-p2wq3", "face-goyue", "Melasma, marks, pores"),
        ("skinconcerns", "face-wrinkles-detection-xjlkr", "Wrinkles"),
        ("james-aiart", "acne-jcvgb", "Acne"),
        ("acne-severity", "clinic-data", "Acne severity"),
    ],
    "nails": [
        ("sergei-artiukhin-ohqpa", "nails-segmentation-yw1zl", "Nail segmentation"),
        ("vincent-y20l1", "fingernail-nqkfm", "Painted and bare nails"),
        ("nailproject", "healthy_nail", "Healthy nails"),
        ("detect-nmkdb", "nail-xaf95", "Nail segmentation"),
        ("aiit-nail-competition", "nail-segmentation-odcgv", "Nail segmentation"),
        ("nsu-sxhmy", "nails-semantic-segmentation", "Nail segmentation"),
        ("personal-projects-jfbag", "nails_segmentation", "Nail segmentation"),
        ("nailproject-padk9", "nail_segmentation", "Nail segmentation"),
        ("fatima-vahabi", "nail-seg-vfa2h", "Nail segmentation"),
        ("vnrvjiet-mnvm9", "nail-images", "Nail conditions"),
        ("nail-acf3b", "nail_disease_own_dataset", "Nail conditions"),
        ("fatima-wmkf6", "nail-segmentation-jrprm", "Nail segmentation"),
    ],
    "hair": [
        ("hairbaldsemanticsegmentation", "hair_baldness", "Scalp and crown"),
        ("umm-alqura-xm8lk", "hairtype", "Curl types incl. 3b and 4A"),
        ("karthik-k", "face-r74ti", "Afro hair, dreadlocks and other styles"),
        ("custom-yolov5-zxmeo", "hair-type", "Straight, wavy, curly"),
        ("flickr", "hair-54plg", "Straight, curly, bald"),
        ("project-kl-wirwir-gg0a3", "hair-style-kpiqp", "Curly, straight, wavy"),
        ("abc-rjytv", "hair_data", "Hair strands"),
        ("itclab", "hair-detection-lkule", "Hair detection"),
        ("hair-oprdr", "mask_segmentation1", "Hair masks"),
        ("manish-kumar-ytl7z", "hair_dataset_updated", "Hair segmentation"),
        # Scalp and hair-health close-ups (found by scout_datasets.ipynb, CC BY 4.0).
        ("sreenithi", "hairproblems-detection", "Good hair, dry scalp, dandruff, oiliness, white hair"),
        ("husnas-workspace", "head-lice-ai", "Healthy scalp and dandruff"),
        ("scalpdet", "scalp-data-iqjst", "Scalp dandruff and oiliness"),
        ("scalpdet", "dandruff-big-chunks", "Dandruff close-ups"),
        ("rooted", "hair-loss-nq8hh-tqveb", "Hair thinning levels"),
        ("major-project-s3rff", "hair-disease-detection-o2ok0", "Scalp conditions"),
    ],
}

NON_COMMERCIAL = ("non-commercial", "noncommercial", "-nc", " nc", "by-nc", "private", "research only")


def project_info(workspace, project, key):
    url = f"https://api.roboflow.com/{workspace}/{project}?api_key={key}"
    with urllib.request.urlopen(url, timeout=60) as response:
        return json.loads(response.read().decode())


def licence_of(info):
    """Roboflow does not document a licence field; look in the usual places."""
    project = info.get("project", info)
    for key in ("license", "licence", "license_name"):
        value = project.get(key) or info.get(key)
        if value:
            return str(value)
    return None


ZENODO_SCALP = "https://zenodo.org/api/records/23433121"


def zenodo_scalp(credits):
    """ScalpHairLabels v1 (CC BY 4.0): 504 scalp close-up tiles cut from CC0 ISIC images."""
    import io
    import urllib.request
    import zipfile

    target = os.path.join("data", "hair", "zenodo__scalphairlabels")
    link = "https://zenodo.org/records/23433121"
    try:
        record = json.loads(urllib.request.urlopen(ZENODO_SCALP, timeout=60).read())
        os.makedirs(target, exist_ok=True)
        for entry in record.get("files", []):
            if "tiles" not in entry["key"]:
                continue
            data = urllib.request.urlopen(entry["links"]["self"], timeout=600).read()
            zipfile.ZipFile(io.BytesIO(data)).extractall(target)
        count = sum(1 for _, _, files in os.walk(target) for f in files if f.lower().endswith((".jpg", ".jpeg", ".png")))
        print(f"OK   hair: ScalpHairLabels v1 ({count} tiles, licence: CC BY 4.0, source photos CC0)")
        credits = [c for c in credits if c["link"] != link]
        credits.append({"area": "hair", "name": "ScalpHairLabels v1", "what": "Scalp close-ups (ISIC, CC0)",
                        "link": link, "licence": "CC BY 4.0", "images": count})
    except Exception as error:
        print(f"FAIL hair: ScalpHairLabels ({str(error)[:120]})")
    return credits


def main():
    key = os.environ.get("ROBOFLOW_API_KEY")
    if not key:
        raise SystemExit("Set ROBOFLOW_API_KEY first (Colab: key icon > Secrets).")
    from roboflow import Roboflow

    rf = Roboflow(api_key=key)
    areas = [a for a in sys.argv[1:] if a in DATASETS] or list(DATASETS)
    os.makedirs("data", exist_ok=True)
    credits_path = os.path.join("data", "credits.json")
    credits = json.load(open(credits_path)) if os.path.exists(credits_path) else []
    for area in areas:
        for workspace, project, what in DATASETS[area]:
            link = f"https://universe.roboflow.com/{workspace}/{project}"
            target = os.path.join("data", area, f"{workspace}__{project}")
            try:
                info = project_info(workspace, project, key)
                licence = licence_of(info)
                if licence and any(word in licence.lower() for word in NON_COMMERCIAL):
                    print(f"SKIP {area}: {project} (licence: {licence})")
                    continue
                versions = rf.workspace(workspace).project(project).versions()
                if not versions:
                    print(f"SKIP {area}: {project} (no downloadable version)")
                    continue
                newest = sorted(versions, key=lambda v: int(str(v.version).split("/")[-1]))[-1]
                if os.path.exists(target):
                    shutil.rmtree(target)
                for fmt in ("coco", "coco-segmentation", "yolov8", "folder"):
                    try:
                        newest.download(fmt, location=target, overwrite=True)
                        break
                    except Exception:
                        continue
                count = sum(
                    1
                    for root, _, files in os.walk(target)
                    for f in files
                    if f.lower().endswith((".jpg", ".jpeg", ".png", ".webp"))
                )
                print(f"OK   {area}: {project} ({count} images, licence: {licence or 'not reported, check the page'})")
                credits = [c for c in credits if c["link"] != link]
                credits.append({"area": area, "name": project, "what": what, "link": link, "licence": licence, "images": count})
            except Exception as error:
                print(f"FAIL {area}: {project} ({str(error)[:120]})")
    if "hair" in areas:
        credits = zenodo_scalp(credits)
    json.dump(credits, open(credits_path, "w"), indent=2)
    print(f"\nCredits written to {credits_path}. Check any 'not reported' licence on its page before shipping.")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile scout_datasets.py
"""Glowtrace dataset scout: searches open sources for skin, hair and nail photos and keeps only
what can be used commercially.

Sources: Hugging Face Hub, Zenodo, Wikimedia Commons, Kaggle (optional key) and Roboflow Universe
(licence and size check of known projects, with your key). Writes scout_report.csv and
scout_report.json: every find with its licence verdict, size and link, best first.

Licence verdicts:
  ok          commercial use allowed (CC0, CC BY, MIT, Apache, BSD, ODC-BY, PDDL, CDLA-Permissive)
  share-alike allowed, but derived data must stay under the same licence (CC BY-SA, ODbL)
  no          non-commercial, no-derivatives, research-only or custom
  unknown     no licence found: treat as no until the owner confirms
"""

import csv
import json
import os
import re
import time
import urllib.parse
import urllib.request

TOPICS = {
    "skin": ["skin texture", "facial skin", "acne", "skin redness", "hyperpigmentation", "skin tone",
             "melasma", "dark circles", "wrinkles", "pores", "face skin dataset", "dark skin",
             "skin of colour", "post-inflammatory hyperpigmentation", "rosacea", "oily skin",
             "dry skin", "under eye", "lips", "makeup", "selfie skin", "face parsing"],
    "hair": ["hair texture", "hair damage", "split ends", "frizz", "curly hair", "afro hair",
             "hair segmentation", "scalp", "dandruff", "hair loss", "hair strands", "hair shine",
             "natural hair", "braids", "box braids", "dreadlocks", "hairline", "traction alopecia",
             "grey hair", "hair length", "curl pattern", "trichoscopy", "hair type"],
    "nails": ["fingernail", "nail segmentation", "nail polish", "nail disease", "nail health",
              "onychomycosis", "nail ridges", "hand segmentation", "manicure", "nail art",
              "acrylic nails", "cuticle", "hand keypoints"],
}

OK = ["cc0", "cc-zero", "cc-by-4", "cc-by-3", "cc-by-2", "cc by 4", "cc by 3", "cc by 2", "public domain",
      "pddl", "mit", "apache", "bsd", "odc-by", "cdla-permissive", "attribution 4.0", "attribution 3.0",
      "attribution 2.0", "cc-by", "cc by"]
SHARE = ["cc-by-sa", "cc by-sa", "sharealike", "share-alike", "odbl"]
NO = ["nc", "noncommercial", "non-commercial", "nd", "noderivatives", "research", "academic",
      "other", "unknown"]


def verdict(licence):
    text = (licence or "").lower().replace("_", "-")
    if not text.strip():
        return "unknown"
    if any(re.search(rf"(^|[^a-z]){w}([^a-z]|$)", text) for w in NO):
        return "no"
    if any(w in text for w in SHARE):
        return "share-alike"
    if any(w in text for w in OK):
        return "ok"
    return "unknown"


def get(url, headers=None):
    request = urllib.request.Request(url, headers={"User-Agent": "dataset-scout/1.0", **(headers or {})})
    with urllib.request.urlopen(request, timeout=30) as response:
        return json.loads(response.read())


finds = {}


def add(area, source, name, link, licence, size, what):
    key = link.lower()
    if key in finds:
        return
    finds[key] = {"area": area, "source": source, "name": name, "link": link, "licence": licence or "",
                  "verdict": verdict(licence), "size": size, "what": (what or "")[:160].replace("\n", " ")}


def huggingface():
    from huggingface_hub import HfApi
    api = HfApi()
    for area, terms in TOPICS.items():
        for term in terms:
            try:
                for d in api.list_datasets(search=term, full=True, limit=40):
                    tags = d.tags or []
                    licence = next((t.split(":", 1)[1] for t in tags if t.startswith("license:")), "")
                    size = next((t.split(":", 1)[1] for t in tags if t.startswith("size_categories:")), "")
                    if not any(t.startswith("modality:image") or t == "task_categories:image-classification"
                               or "image" in t for t in tags):
                        continue
                    add(area, "Hugging Face", d.id, f"https://huggingface.co/datasets/{d.id}", licence, size, term)
            except Exception as error:
                print("  Hugging Face", term, error)


IMAGEY = re.compile(r"\b(image|images|photo|photos|photograph|dermoscop|segmentation|annotated)\b")
IRRELEVANT = re.compile(r"\b(eeg|meg|mri|ct scan|spectrogram|genom|rna|protein|sequenc)")


def zenodo():
    for area, terms in TOPICS.items():
        for term in terms:
            q = urllib.parse.quote(f'title:"{term}" AND (images OR photographs OR photos)')
            try:
                data = get(f"https://zenodo.org/api/records?q={q}&type=dataset&size=25")
                for hit in data.get("hits", {}).get("hits", []):
                    meta = hit.get("metadata", {})
                    text = (meta.get("title", "") + " " + meta.get("description", "")).lower()
                    if IRRELEVANT.search(text) or not IMAGEY.search(text):
                        continue
                    licence = (meta.get("license") or {}).get("id", "")
                    files = hit.get("files") or []
                    size = f"{sum(f.get('size', 0) for f in files) / 1e9:.1f} GB" if files else ""
                    add(area, "Zenodo", meta.get("title", "")[:90], hit.get("links", {}).get("self_html", ""),
                        licence, size, term)
                time.sleep(0.5)
            except Exception as error:
                print("  Zenodo", term, error)


COMMONS = {"skin": ["Human skin", "Acne", "Freckles"], "hair": ["Human hair", "Afro-textured hair",
           "Curly hair", "Dreadlocks", "Hair braids"], "nails": ["Fingernails", "Nail polish", "Manicure"]}


def wikimedia():
    for area, categories in COMMONS.items():
        for category in categories:
            title = urllib.parse.quote(f"Category:{category}")
            try:
                data = get(f"https://commons.wikimedia.org/w/api.php?action=query&prop=categoryinfo&titles={title}&format=json")
                page = next(iter(data["query"]["pages"].values()))
                files = page.get("categoryinfo", {}).get("files", 0)
                add(area, "Wikimedia Commons", f"Category:{category}",
                    f"https://commons.wikimedia.org/wiki/Category:{urllib.parse.quote(category)}",
                    "CC BY / CC BY-SA / public domain (per file)", f"{files} files",
                    "each file carries its own licence: keep only CC0, CC BY and public domain")
                time.sleep(1.5)
            except Exception as error:
                print("  Wikimedia", category, error)


def figshare():
    for area, terms in TOPICS.items():
        for term in terms:
            try:
                body = json.dumps({"search_for": f'"{term}"', "item_type": 3, "page_size": 20}).encode()
                request = urllib.request.Request("https://api.figshare.com/v2/articles/search", data=body,
                                                 headers={"Content-Type": "application/json", "User-Agent": "dataset-scout/1.0"})
                with urllib.request.urlopen(request, timeout=30) as response:
                    hits = json.loads(response.read())
                for hit in hits:
                    title = hit.get("title", "")
                    if IRRELEVANT.search(title.lower()):
                        continue
                    detail = get(f"https://api.figshare.com/v2/articles/{hit['id']}")
                    text = (title + " " + detail.get("description", "")).lower()
                    if not IMAGEY.search(text):
                        continue
                    size = sum(f.get("size", 0) for f in detail.get("files", []))
                    if size < 20e6:
                        continue
                    add(area, "Figshare", title[:90], hit.get("url_public_html", ""),
                        (detail.get("license") or {}).get("name", ""), f"{size / 1e9:.1f} GB", term)
                time.sleep(0.5)
            except Exception as error:
                print("  Figshare", term, error)


def kaggle():
    if not (os.environ.get("KAGGLE_USERNAME") and os.environ.get("KAGGLE_KEY")):
        print("  Kaggle skipped (add KAGGLE_USERNAME and KAGGLE_KEY to Colab Secrets to include it)")
        return
    from kaggle.api.kaggle_api_extended import KaggleApi
    api = KaggleApi()
    api.authenticate()
    for area, terms in TOPICS.items():
        for term in terms:
            try:
                for d in api.dataset_list(search=term, file_type="all")[:20]:
                    add(area, "Kaggle", d.ref, f"https://www.kaggle.com/datasets/{d.ref}",
                        getattr(d, "licenseName", "") or "", getattr(d, "totalBytes", "") or "", d.title)
            except Exception as error:
                print("  Kaggle", term, error)


# Roboflow projects found by hand; the scout checks each one's licence and size with your key.
ROBOFLOW = {
    "hair": ["scalpdet/scalp-data-iqjst", "scalpdet/dandruff-big-chunks", "husnas-workspace/head-lice-ai",
             "nayab-waris-hfzbj/hair-and-scalp-disease", "nayab-waris-hfzbj/hair_and_scalp_disease",
             "rooted/hair-loss-nq8hh-tqveb", "sreenithi/hairproblems-detection", "itclab/hair-detection-lkule",
             "project-teqqx/hair-disease-detection", "major-project-s3rff/hair-disease-detection-o2ok0",
             "inferenz/hair-and-bald"],
    "nails": ["vincent-y20l1/upper-body-fingernail", "nsu-sxhmy/nails-semantic-segmentation",
              "linkerverse-xzpl7/nail_instance_segmentation", "fatima-wmkf6/nail-segmentation-jrprm",
              "rizqi-oleec/nail-segmenter", "nailproject-padk9/nail_segmentation"],
}


def roboflow():
    key = os.environ.get("ROBOFLOW_API_KEY")
    if not key:
        print("  Roboflow skipped (no ROBOFLOW_API_KEY)")
        return
    for area, projects in ROBOFLOW.items():
        for slug in projects:
            try:
                data = get(f"https://api.roboflow.com/{slug}?api_key={key}")
                project = data.get("project", {})
                add(area, "Roboflow", slug, f"https://universe.roboflow.com/{slug}", project.get("license", ""),
                    f"{project.get('images', '')} images", ", ".join((project.get("classes") or {}).keys())[:160])
            except Exception as error:
                add(area, "Roboflow", slug, f"https://universe.roboflow.com/{slug}", "", "", f"not reachable: {error}")


if __name__ == "__main__":
    for name, step in [("Hugging Face", huggingface), ("Zenodo", zenodo), ("Figshare", figshare),
                       ("Wikimedia Commons", wikimedia), ("Kaggle", kaggle), ("Roboflow", roboflow)]:
        print("Searching", name, flush=True)
        step()
    order = {"ok": 0, "share-alike": 1, "unknown": 2, "no": 3}
    rows = sorted(finds.values(), key=lambda r: (order[r["verdict"]], r["area"], r["source"], r["name"]))
    with open("scout_report.csv", "w", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=list(rows[0].keys()) if rows else ["area"])
        writer.writeheader()
        writer.writerows(rows)
    json.dump(rows, open("scout_report.json", "w"), indent=1)
    for v in order:
        print(f"{v:12s}", sum(r["verdict"] == v for r in rows))
    print("Top commercial-use finds:")
    for r in [r for r in rows if r["verdict"] == "ok"][:40]:
        print(f"  [{r['area']}] {r['source']}: {r['name']} ({r['licence']}, {r['size']})")


In [ ]:
%%writefile collect_datasets.py
"""Download every usable dataset from scout_report.json into data/<area>/ and record credits.

Takes only finds whose licence allows commercial use ("ok"), that look relevant to their area and
are a sensible size. Roboflow sets come through the Roboflow API (your key); Zenodo and Figshare
files are fetched directly and unzipped. Hugging Face sets are skipped unless --include-hf, because
most have no record of where their photos came from. Every download is marked "provenance:
check" in data/credits.json until Claude has traced where its photos came from.

Usage: python collect_datasets.py [--include-hf] [--max-gb 5]
"""

import io
import json
import os
import re
import sys
import urllib.request
import zipfile

MAX_GB = float(sys.argv[sys.argv.index("--max-gb") + 1]) if "--max-gb" in sys.argv else 5.0
INCLUDE_HF = "--include-hf" in sys.argv
WORDS = {
    "skin": r"skin|face|facial|acne|pigment|melasma|lesion|derm|wrinkle|pore|selfie|lip|makeup",
    "hair": r"hair|scalp|dandruff|braid|loc|curl|afro|trich|alopecia|bald",
    "nails": r"nail|finger|hand|manicure|cuticle|polish",
}


def slug(text):
    return re.sub(r"[^a-z0-9]+", "-", text.lower()).strip("-")[:60]


def gigabytes(size):
    match = re.match(r"([\d.]+)\s*GB", size or "")
    return float(match.group(1)) if match else None


def fetch(url, timeout=900):
    request = urllib.request.Request(url, headers={"User-Agent": "glowtrace-collect/1.0"})
    with urllib.request.urlopen(request, timeout=timeout) as response:
        return response.read()


def unpack(data, name, target):
    os.makedirs(target, exist_ok=True)
    if name.lower().endswith(".zip"):
        zipfile.ZipFile(io.BytesIO(data)).extractall(target)
    elif name.lower().endswith((".jpg", ".jpeg", ".png", ".webp")):
        open(os.path.join(target, os.path.basename(name)), "wb").write(data)


def count_images(target):
    return sum(1 for _, _, files in os.walk(target) for f in files
               if f.lower().endswith((".jpg", ".jpeg", ".png", ".webp")))


def zenodo(link, target):
    record = json.loads(fetch(f"https://zenodo.org/api/records/{link.rstrip('/').split('/')[-1]}"))
    for entry in record.get("files", []):
        if entry["key"].lower().endswith((".zip", ".jpg", ".jpeg", ".png")):
            unpack(fetch(entry["links"]["self"]), entry["key"], target)


def figshare(link, target):
    article = link.rstrip("/").split("/")[-1]
    detail = json.loads(fetch(f"https://api.figshare.com/v2/articles/{article}"))
    for entry in detail.get("files", []):
        if entry["name"].lower().endswith((".zip", ".jpg", ".jpeg", ".png")):
            unpack(fetch(entry["download_url"]), entry["name"], target)


def roboflow(name, target):
    from roboflow import Roboflow
    workspace, project = name.split("/")
    versions = Roboflow(api_key=os.environ["ROBOFLOW_API_KEY"]).workspace(workspace).project(project).versions()
    if not versions:
        raise RuntimeError("no downloadable version")
    newest = sorted(versions, key=lambda v: int(str(v.version).split("/")[-1]))[-1]
    for fmt in ("coco", "coco-segmentation", "yolov8", "folder"):
        try:
            newest.download(fmt, location=target, overwrite=True)
            return
        except Exception:
            continue
    raise RuntimeError("no format downloaded")


def huggingface(name, target, limit=3000):
    from datasets import load_dataset
    os.makedirs(target, exist_ok=True)
    stream = load_dataset(name, split="train", streaming=True)
    saved = 0
    for row in stream:
        image = next((v for v in row.values() if hasattr(v, "save")), None)
        if image is None:
            continue
        image.convert("RGB").save(os.path.join(target, f"{saved:05d}.jpg"), quality=92)
        saved += 1
        if saved >= limit:
            break


def main():
    rows = json.load(open("scout_report.json"))
    os.makedirs("data", exist_ok=True)
    credits_path = os.path.join("data", "credits.json")
    credits = json.load(open(credits_path)) if os.path.exists(credits_path) else []
    known = {c["link"] for c in credits}
    for row in rows:
        if row["verdict"] != "ok" or row["link"] in known:
            continue
        if not re.search(WORDS[row["area"]], (row["name"] + " " + row["what"]).lower()):
            print(f"skip (not about {row['area']}): {row['name'][:70]}")
            continue
        size = gigabytes(row["size"])
        if size is not None and size > MAX_GB:
            print(f"skip (too big, {row['size']}): {row['name'][:70]}")
            continue
        if row["source"] == "Hugging Face" and not INCLUDE_HF:
            print(f"skip (Hugging Face, provenance unknown; use --include-hf): {row['name']}")
            continue
        target = os.path.join("data", row["area"], f"{slug(row['source'])}__{slug(row['name'])}")
        try:
            {"Zenodo": zenodo, "Figshare": figshare}.get(row["source"], lambda *_: None)(row["link"], target)
            if row["source"] == "Roboflow":
                roboflow(row["name"], target)
            if row["source"] == "Hugging Face":
                huggingface(row["name"], target)
            images = count_images(target) if os.path.exists(target) else 0
            if images == 0:
                print(f"none  {row['source']}: {row['name'][:70]} (no images in the files)")
                continue
            print(f"OK    {row['source']}: {row['name'][:70]} ({images} images)")
            credits.append({"area": row["area"], "name": row["name"], "what": row["what"], "link": row["link"],
                            "licence": row["licence"], "images": images, "provenance": "check"})
            json.dump(credits, open(credits_path, "w"), indent=2)
        except Exception as error:
            print(f"FAIL  {row['source']}: {row['name'][:70]} ({str(error)[:100]})")
    print(f"\nCredits in {credits_path}. Send it to Claude so each new set's photo sources can be checked.")


if __name__ == "__main__":
    main()


In [ ]:
# 2. The curated sets (Roboflow and ScalpHairLabels). Already-downloaded sets are refreshed.
!python fetch_datasets.py

In [ ]:
# 3. Search every source for more.
!python scout_datasets.py

In [ ]:
# 4. Download every find that's allowed, relevant and under 5 GB.
!python collect_datasets.py --max-gb 5

In [ ]:
# 5. Summary, then download the two files for Claude.
import json, glob
from google.colab import files
credits = json.load(open('data/credits.json'))
for area in ['skin', 'hair', 'nails']:
    sets = [c for c in credits if c['area'] == area]
    print(f"{area}: {len(sets)} sets, {sum(c.get('images', 0) for c in sets):,} images")
files.download('data/credits.json')
files.download('scout_report.csv')